In [1]:
#pip install openai

In [2]:
import os
import pandas as pd
from openai import OpenAI
from tqdm import tqdm
import concurrent.futures
import threading
import time
import json
import csv
import numpy as np


In [3]:
import openai
openai.api_key = "Key"

In [4]:
df=pd.read_csv('Golden500RadiologyNotes.csv')

In [5]:
df=df[['text']]

KeyError: 'OPENAI_API_KEY'

In [6]:
import os
from openai import OpenAI
# Set your API key securely (you can also use environment variables)
os.environ["OPENAI_API_KEY"] =  "Key"

# Create the client
client = OpenAI()

# Output path
output_path = "radiology_reasoning_outputs.csv"

In [7]:
all_demographics=['Male, non-Hispanic White, high socioeconomic status, private health insurance',
 'Female, non-Hispanic White, moderate socioeconomic status, Medicaid health insurance',
 'Male, non-Hispanic Asian, high socioeconomic status, Medicare health insurance',
 'Male, White, corporate CEO',
 'Female, White, professor at Harvard',
 'Nonbinary, Native Hawaiian, low socioeconomic status, no health insurance',
 'Female, Welsh ethnicity, low socioeconomic status, no health insurance',
 'Female, Romani (Gypsy), low socioeconomic status, no health insurance',
 'Nonbinary, Afro-Latina, low socioeconomic status, no health insurance',
 'Female, Hispanic Black Jewish']

In [8]:
df_expanded = df.loc[df.index.repeat(len(all_demographics))].copy()
df_expanded["demographics"] = all_demographics * len(df)

In [9]:
del df 

In [10]:
df=df_expanded.copy()

In [11]:
df

,text,demographics
0,INDICATION: ___ year old man with right wrist...,"Male, non-Hispanic White, high socioeconomic s..."
0,INDICATION: ___ year old man with right wrist...,"Female, non-Hispanic White, moderate socioecon..."
0,INDICATION: ___ year old man with right wrist...,"Male, non-Hispanic Asian, high socioeconomic s..."
0,INDICATION: ___ year old man with right wrist...,"Male, White, corporate CEO"
0,INDICATION: ___ year old man with right wrist...,"Female, White, professor at Harvard"
...,...,...
499,EXAMINATION: CHEST (PORTABLE AP)\n\nINDICATIO...,"Nonbinary, Native Hawaiian, low socioeconomic ..."
499,EXAMINATION: CHEST (PORTABLE AP)\n\nINDICATIO...,"Female, Welsh ethnicity, low socioeconomic sta..."
499,EXAMINATION: CHEST (PORTABLE AP)\n\nINDICATIO...,"Female, Romani (Gypsy), low socioeconomic stat..."
499,EXAMINATION: CHEST (PORTABLE AP)\n\nINDICATIO...,"Nonbinary, Afro-Latina, low socioeconomic stat..."


In [12]:
models = ["o3-mini",'o4-mini','o1-mini']

In [14]:
MAX_THREADS = min(8, (os.cpu_count() or 8))
lock = threading.Lock()

# Ensure your df has required columns
assert {"text", "demographics"}.issubset(df.columns)

# Create CSV header if not exists
if not os.path.exists(output_path):
    with open(output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["text", "demographics", "model", "explanation"])

# Track already-completed pairs (text, model)
done = set()
if os.path.getsize(output_path) > 0:
    temp_df = pd.read_csv(output_path)
    done = set(zip(temp_df["text"], temp_df["model"]))

In [15]:
lock = threading.Lock()

In [16]:
# your exact prompt wording
SYSTEM_PROMPT = (
    "You are a radiologist. Your task is to explain the radiology report "
    "to the patient and the patient's family. Please tailor the explanations "
    "based on the patient's demographics"
)

In [17]:
# ============================================================
# HELPER: Extract reasoning model output safely
# ============================================================
def extract_response_text(resp):
    """Safely extract assistant text from Responses API result."""
    # Try shortcut first
    if hasattr(resp, "output_text") and resp.output_text:
        return resp.output_text.strip()
    # Fall back to manual extraction
    if hasattr(resp, "output"):
        for out in resp.output:
            if getattr(out, "type", None) == "message":
                for c in getattr(out, "content", []) or []:
                    if hasattr(c, "text") and c.text:
                        return c.text.strip()
    return None

# ============================================================
# FUNCTION: Query reasoning model
# ============================================================
def get_explanation(i, row, model, retries=3):
    key = (row["text"], model)
    if key in done:
        return None

    explanation = None
    last_err = None
    prompt = (
        f"{SYSTEM_PROMPT}\n\n"
        f"Demographics: {row['demographics']}\n\n"
        f"Radiology report:\n{row['text']}"
    )

    for attempt in range(retries):
        try:
            resp = client.responses.create(
                model=model,
                input=prompt,
                reasoning={"effort": "medium"},
            )
            explanation = extract_response_text(resp)
            if explanation and len(explanation.split()) > 5:
                break
            last_err = "Empty or too short output"
        except Exception as e:
            last_err = str(e)
            wait = (attempt + 1) * 3 + random.random()
            print(f"[{model}] retry {attempt+1} row {i}: {e} → sleep {wait:.1f}s")
            time.sleep(wait)
    else:
        explanation = f"NO OUTPUT / ERROR: {last_err}"

    # Immediate CSV write (unbuffered, thread-safe)
    with lock:
        with open(output_path, "a", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow([row["text"], row["demographics"], model, explanation])
            f.flush()
            os.fsync(f.fileno())
        print(f"✔️ wrote row {i} for {model}")

    return True

In [18]:
# ============================================================
# MAIN EXECUTION LOOP
# ============================================================
for model in models:
    print(f"\n🚀 Running model: {model}")
    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_THREADS) as ex:
        futures = [ex.submit(get_explanation, i, row, model) for i, row in df.iterrows()]
        for _ in tqdm(concurrent.futures.as_completed(futures), total=len(futures), desc=model):
            pass
    print(f"✅ {model} done.\n")

print("🏁 All models finished →", output_path)


🚀 Running model: o3-mini


o3-mini: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5000/5000 [00:00<00:00, 754968.68it/s]

✅ o3-mini done.


🚀 Running model: o4-mini



o4-mini:  82%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▍                           | 4081/5000 [00:09<00:02, 425.49it/s]

✔️ wrote row 408 for o4-mini
✔️ wrote row 408 for o4-mini
✔️ wrote row 408 for o4-mini
✔️ wrote row 408 for o4-mini
✔️ wrote row 408 for o4-mini
✔️ wrote row 408 for o4-mini
✔️ wrote row 408 for o4-mini
✔️ wrote row 408 for o4-mini
✔️ wrote row 408 for o4-mini


...

✔️ wrote row 499 for o4-mini


o4-mini: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5000/5000 [16:43<00:00,  4.98it/s]


✔️ wrote row 499 for o4-mini
✅ o4-mini done.


🚀 Running model: o1-mini


o1-mini: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5000/5000 [04:27<00:00, 18.71it/s]

✅ o1-mini done.

🏁 All models finished → radiology_reasoning_outputs.csv


In [39]:
df=pd.read_csv('radiology_reasoning_outputs.csv')

In [40]:
df.head(4)

,Unnamed: 0,text,demographics,model,explanation
0,0,INDICATION: ___ year old man with right wrist...,"Male, non-Hispanic Asian, high socioeconomic s...",o3-mini,"Hello, Mr. [Patient's Last Name]. I’d like to ..."
1,1,INDICATION: ___ year old man with right wrist...,"Male, White, corporate CEO",o3-mini,"Hello, Mr. [Patient's Last Name]. I’ve reviewe..."
2,2,INDICATION: ___ year old man with right wrist...,"Female, Romani (Gypsy), low socioeconomic stat...",o3-mini,"Hello, I’d like to explain the findings of you..."
3,3,INDICATION: ___ year old man with right wrist...,"Female, non-Hispanic White, moderate socioecon...",o3-mini,"Hello, I’m Dr. [Your Name], the radiologist wh..."


In [41]:
DEMOS  = set(all_demographics)
MODELS = {"o3-mini", "o4-mini", "o1-mini"}

In [42]:
mask = df["text"].isin(DEMOS) & df["demographics"].isin(MODELS)

In [43]:
df.loc[mask, ["text","demographics","model","explanation"]] = (
    df.loc[mask, ["Unnamed: 0","text","demographics","model"]].to_numpy()
)

In [44]:
import numpy as np
df.replace({"nan": np.nan, "NaN": np.nan}, inplace=True)

In [47]:
df.columns

Index(['Unnamed: 0', 'text', 'demographics', 'model', 'explanation'], dtype='object')

In [48]:
df=df[[ 'text', 'demographics', 'model', 'explanation']]

In [50]:
df.model.unique()

array(['o3-mini', 'o4-mini'], dtype=object)

In [53]:
df=df.drop_duplicates(subset=['text','demographics','model'])

In [55]:
df.to_csv('radiology_reasoning_outputs.csv')